# WiDS Global Datathon 2026 - Hybrid Survival Ensemble


In [1]:

import os, json, math, warnings
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # "full" or "fast"
DO_OOF = True
USE_LEGACY_BLEND = False
LEGACY_BLEND_WEIGHT = 0.10
LEGACY_FILES = []   # Optional: add paths to your old submission CSVs later if you want an extra test-only blend hook.

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")
FAST = RUN_MODE != "full"

def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    if os.path.isdir("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            files = set(files)
            if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files):
                return root
    raise FileNotFoundError("Competition data directory not found.")

DATA_DIR = find_data_dir()

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR =", DATA_DIR)
print("TRAIN =", train_df.shape, "TEST =", test_df.shape)

def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1.0)
    speed = r["closing_speed_m_per_h"]
    speed_pos = speed.clip(lower=0.0)
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"].clip(lower=0.0)
    area_growth_abs = r["area_growth_abs_0_5h"].clip(lower=0.0)
    area_growth = r["area_growth_rate_ha_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0.0)
    align = r["alignment_abs"]
    align_signed = r["alignment_cos"]
    fire_radius_m = np.sqrt(np.maximum(area_first, 0.0) * 10000.0 / np.pi)
    eff_close = speed_pos + radial

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["inv_distance"] = 1.0 / (r["dist_km"] + 0.10)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"] = np.sqrt(dist)
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["fire_radius_km"] = fire_radius_m / 1000.0
    r["radius_to_dist"] = fire_radius_m / dist
    r["area_to_dist_ratio"] = area_first / (r["dist_km"] + 0.10)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)
    r["growth_to_dist_ratio"] = area_growth_abs / (r["dist_km"] + 0.10)

    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(speed_pos > 1e-6, dist / speed_pos, 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 1e-6, dist / eff_close, 9999.0).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])
    r["eta_vs_12"] = 12.0 / (12.0 + r["eta_effective"])
    r["eta_vs_24"] = 24.0 / (24.0 + r["eta_effective"])
    r["eta_vs_48"] = 48.0 / (48.0 + r["eta_effective"])

    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_signed"] = align_signed * eff_close / np.log1p(dist)
    r["fire_urgency"] = perims * eff_close
    r["growth_intensity"] = np.maximum(area_growth, 0.0) * perims
    r["effective_alignment"] = eff_close * align
    r["speed_alignment"] = speed * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0.0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0.0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0.0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0.0)
    r["fit_weighted_close"] = r["slope_toward"] * (0.25 + r["dist_fit_r2_0_5h"])
    r["fit_weighted_threat"] = r["threat_score_eff"] * (0.25 + r["dist_fit_r2_0_5h"])
    r["resolution_weighted_threat"] = r["threat_score_eff"] * np.maximum(r["dt_first_last_0_5h"], 0.10)

    r["toward_zone_speed"] = np.maximum(r["along_track_speed"], 0.0)
    r["away_zone_speed"] = np.maximum(-r["along_track_speed"], 0.0)
    r["cross_to_along_ratio"] = np.abs(r["cross_track_component"]) / (np.abs(r["along_track_speed"]) + 1.0)

    r["zone_3km"] = (dist < 3000.0).astype(float)
    r["zone_5km"] = (dist < 5000.0).astype(float)
    r["zone_8km"] = (dist < 8000.0).astype(float)
    r["zone_10km"] = (dist < 10000.0).astype(float)
    r["zone_far"] = (dist >= 10000.0).astype(float)

    r["high_alignment"] = (align > 0.80).astype(float)
    r["has_growth"] = (area_growth_abs > 1e-9).astype(float)
    r["has_movement"] = (perims > 1).astype(float)
    r["low_res_or_static"] = ((r["low_temporal_resolution_0_5h"] == 1) | (perims <= 1)).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2.0 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2.0 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2.0 * np.pi * (r["event_start_month"] - 1.0) / 12.0)
    r["month_cos"] = np.cos(2.0 * np.pi * (r["event_start_month"] - 1.0) / 12.0)
    r["dow_sin"] = np.sin(2.0 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2.0 * np.pi * r["event_start_dayofweek"] / 7.0)

    def pct_rank(values, ref_values):
        ref_values = np.asarray(ref_values, dtype=float)
        if ref_values.size == 0:
            return np.zeros(len(values), dtype=float)
        ref_sorted = np.sort(ref_values)
        return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / ref_sorted.size

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1.0).values
    ref_eff = (ref["closing_speed_m_per_h"].clip(lower=0.0).values +
               ref["radial_growth_rate_m_per_h"].clip(lower=0.0).values)
    ref_threat = ref["alignment_abs"].values * ref_eff / np.log1p(ref_dist)
    ref_eta = np.log1p(np.where(ref_eff > 1e-6, ref_dist / ref_eff, 9999.0).clip(max=9999.0))

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)
    r["eta_rank_global"] = pct_rank(r["log_eta_effective"].values, ref_eta)

    ref_near = ref_dist < 5000.0
    cur_near = dist.values < 5000.0
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r["threat_score_eff"].values[~cur_near], far_ref_threat)
    r["near_speed_rank"] = near_speed_rank
    r["far_threat_rank"] = far_threat_rank

    drop_cols = [
        "relative_growth_0_5h",
        "closing_speed_abs_m_per_h",
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return r

train_proc = create_features(train_df)
test_proc = create_features(test_df, fit_df=train_df)

RAW_FEATURES = [c for c in train_proc.columns if c not in ["event_id", "event", "time_to_hit_hours"]]

GLOBAL_FEATURES = [
    "dist_min_ci_0_5h", "dist_km", "log_distance", "inv_distance", "inv_distance_sq", "sqrt_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h", "effective_closing_speed",
    "alignment_abs", "alignment_cos", "threat_score_eff", "threat_score_signed",
    "eta_effective", "log_eta_effective", "eta_vs_24", "eta_vs_48",
    "area_to_dist_ratio", "log_area_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "dt_first_last_0_5h", "low_temporal_resolution_0_5h",
    "projected_advance_pos", "slope_toward", "accel_toward",
    "fit_weighted_close", "fit_weighted_threat", "resolution_weighted_threat",
    "dist_fit_r2_0_5h", "near_speed_rank", "far_threat_rank",
    "dist_rank_global", "eff_rank_global", "threat_rank_global", "eta_rank_global",
    "high_alignment", "has_growth", "has_movement", "low_res_or_static",
    "is_afternoon", "is_summer", "hour_sin", "hour_cos", "month_sin", "month_cos",
    "log1p_area_first", "log1p_growth", "area_growth_rate_ha_per_h",
    "spread_bearing_sin", "spread_bearing_cos", "toward_zone_speed", "cross_to_along_ratio",
]
GLOBAL_FEATURES = [c for c in GLOBAL_FEATURES if c in train_proc.columns]

NEAR_FEATURES = [
    "dist_km", "inv_distance", "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "effective_closing_speed", "alignment_abs", "threat_score_eff",
    "eta_effective", "log_eta_effective", "near_speed_rank",
    "fire_radius_km", "area_to_dist_ratio", "projected_advance_pos",
    "num_perimeters_0_5h", "dt_first_last_0_5h", "low_temporal_resolution_0_5h",
    "high_alignment", "has_growth", "hour_sin", "hour_cos", "month_sin", "month_cos",
]
NEAR_FEATURES = [c for c in NEAR_FEATURES if c in train_proc.columns]

FAR_FEATURES = [
    "dist_km", "log_distance", "inv_distance", "effective_closing_speed",
    "alignment_abs", "threat_score_eff", "threat_score_signed",
    "eta_effective", "log_eta_effective", "far_threat_rank",
    "fit_weighted_close", "slope_toward", "accel_toward",
    "fire_radius_km", "area_to_dist_ratio",
    "num_perimeters_0_5h", "dt_first_last_0_5h", "low_temporal_resolution_0_5h",
    "dist_fit_r2_0_5h", "is_summer", "month_sin", "month_cos",
]
FAR_FEATURES = [c for c in FAR_FEATURES if c in train_proc.columns]

LINEAR_FEATURES = [
    "dist_km", "log_distance", "inv_distance", "inv_distance_sq",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h", "effective_closing_speed",
    "alignment_abs", "alignment_cos", "threat_score_eff", "threat_score_signed",
    "eta_effective", "log_eta_effective", "eta_vs_12", "eta_vs_24", "eta_vs_48",
    "area_to_dist_ratio", "log_area_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "dt_first_last_0_5h", "low_temporal_resolution_0_5h",
    "dist_fit_r2_0_5h", "near_speed_rank", "far_threat_rank",
    "projected_advance_pos", "slope_toward", "accel_toward",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_proc.columns]

SURV_FEATURES = sorted(set(NEAR_FEATURES + FAR_FEATURES + [
    "dist_rank_global", "eff_rank_global", "threat_rank_global", "eta_rank_global",
    "high_alignment", "has_growth", "low_res_or_static",
]))
SURV_FEATURES = [c for c in SURV_FEATURES if c in train_proc.columns]

X_global_train = train_proc[GLOBAL_FEATURES]
X_global_test = test_proc[GLOBAL_FEATURES]
X_near_train = train_proc[NEAR_FEATURES]
X_near_test = test_proc[NEAR_FEATURES]
X_far_train = train_proc[FAR_FEATURES]
X_far_test = test_proc[FAR_FEATURES]
X_linear_train = train_proc[LINEAR_FEATURES]
X_linear_test = test_proc[LINEAR_FEATURES]
X_surv_train = train_proc[SURV_FEATURES]
X_surv_test = test_proc[SURV_FEATURES]

time_values = train_df["time_to_hit_hours"].values.astype(float)
event_values = train_df["event"].values.astype(int)
dist_train_m = train_proc["dist_min_ci_0_5h"].values.astype(float)
dist_test_m = test_proc["dist_min_ci_0_5h"].values.astype(float)

def compute_c_index(time, event, risk):
    n = len(time)
    conc = 0.0
    comp = 0.0
    risk = np.asarray(risk, dtype=float)
    for i in range(n):
        if event[i] != 1:
            continue
        ti = time[i]
        for j in range(n):
            if i == j or ti >= time[j]:
                continue
            comp += 1.0
            if risk[i] > risk[j]:
                conc += 1.0
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def compute_brier(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    prob = np.clip(np.asarray(prob)[valid], 0.0, 1.0)
    return float(np.mean((prob - y_true) ** 2))

def compute_hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * np.asarray(p24) + 0.4 * np.asarray(p48) + 0.3 * np.asarray(p72)
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    hybrid = 0.3 * c_idx + 0.7 * (1.0 - wb)
    return hybrid, c_idx, wb, b24, b48, b72

def enforce_monotonicity(preds):
    out = np.clip(np.asarray(preds, dtype=float), 0.0, 1.0)
    for i in range(1, out.shape[1]):
        out[:, i] = np.maximum(out[:, i], out[:, i - 1])
    return out

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    running = 1.0
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            running *= (1.0 - cens / at_risk)
        surv[i] = max(running, 0.01)

    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0

    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
        else:
            weights[i] = 0.0
    return weights

class ConstantCalibrator:
    def __init__(self, value):
        self.value = float(value)
    def predict(self, x):
        x = np.asarray(x)
        return np.full(x.shape[0], self.value, dtype=float)

def fit_calibrator(scores, times, events, horizon):
    scores = np.asarray(scores, dtype=float)
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    valid = ~((events == 0) & (times < horizon))
    y = ((events == 1) & (times <= horizon)).astype(int)
    if valid.sum() < 6 or len(np.unique(y[valid])) < 2:
        return ConstantCalibrator(float(y[valid].mean()) if valid.sum() else 0.5)
    model = IsotonicRegression(out_of_bounds="clip")
    model.fit(scores[valid], y[valid])
    return model

def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model
    except Exception:
        model.fit(X, y)
        return model

def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else np.asarray(p)

def soft_gate(dist_m, center, width):
    z = np.clip((np.asarray(dist_m, dtype=float) - center) / max(width, 1.0), -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(z))

def stabilize(pred, anchor, alpha):
    return np.clip(alpha * np.asarray(pred, dtype=float) + (1.0 - alpha) * np.asarray(anchor, dtype=float), 0.0, 1.0)

def repeated_isotonic(train_signal, test_signal, y, mask, seeds, n_splits=5):
    n_train = len(train_signal)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)
    valid_idx = np.where(mask)[0]
    Xv = np.asarray(train_signal)[valid_idx]
    yv = y[valid_idx]

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv.reshape(-1, 1), yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            cal = fit_calibrator(np.asarray(train_signal)[tr_idx], time_values[tr_idx], event_values[tr_idx], int(np.unique(np.where(y[tr_idx] == 1, 1, 0)).sum() + 1))
            ir = IsotonicRegression(out_of_bounds="clip")
            if len(np.unique(y[tr_idx])) < 2:
                oof[va_idx] += y[tr_idx].mean()
                cnt[va_idx] += 1.0
                continue
            ir.fit(np.asarray(train_signal)[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(np.asarray(train_signal)[va_idx])
            cnt[va_idx] += 1.0

        ir_full = IsotonicRegression(out_of_bounds="clip")
        if len(np.unique(yv)) < 2:
            fill_val = float(yv.mean())
            full_fill += fill_val
            test_pred += fill_val / len(seeds)
        else:
            ir_full.fit(Xv, yv)
            full_fill += ir_full.predict(np.asarray(train_signal))
            test_pred += ir_full.predict(np.asarray(test_signal)) / len(seeds)

    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz] / len(seeds)
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def repeated_binary_model(X_train, X_test, y, mask, build_model, seeds, n_splits=5, horizon=None, use_ipcw=False, train_select_mask=None):
    n_train = len(X_train)
    n_test = len(X_test)
    valid_idx = np.where(mask)[0]
    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]

            if train_select_mask is not None:
                tr_idx_candidate = tr_idx[np.asarray(train_select_mask, dtype=bool)[tr_idx]]
                if len(np.unique(y[tr_idx_candidate])) >= 2:
                    tr_idx = tr_idx_candidate

            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        full_idx = valid_idx
        if train_select_mask is not None:
            full_idx_candidate = full_idx[np.asarray(train_select_mask, dtype=bool)[full_idx]]
            if len(np.unique(y[full_idx_candidate])) >= 2:
                full_idx = full_idx_candidate

        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[full_idx], event_values[full_idx], horizon)
        fit_with_optional_weight(model_full, X_train.iloc[full_idx], y[full_idx], sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def make_strat_labels(time, event):
    bins = pd.qcut(np.where(event == 1, np.minimum(time, 72.0), 72.0 + np.minimum(time, 72.0)), q=5, duplicates="drop", labels=False)
    labels = event.astype(str) + "_" + pd.Series(bins).astype(str)
    codes, _ = pd.factorize(labels)
    return codes

STRAT_LABELS = make_strat_labels(time_values, event_values)

def aft_cdf_from_margin(margin, horizon, distribution, scale):
    z = (np.log(np.maximum(float(horizon), 1e-12)) - np.asarray(margin, dtype=float)) / float(scale)
    if distribution == "normal":
        return norm.cdf(z)
    if distribution == "logistic":
        return expit(z)
    if distribution == "extreme":
        return 1.0 - np.exp(-np.exp(z))
    raise ValueError(f"Unsupported AFT distribution: {distribution}")

def repeated_xgb_cox(X_train, X_test, params, seeds, horizons=(12, 24, 48), n_splits=5):
    labels = np.where(event_values == 1, time_values, -time_values)
    oof = {h: np.zeros(len(X_train), dtype=float) for h in horizons}
    cnt = np.zeros(len(X_train), dtype=float)
    test_pred = {h: np.zeros(len(X_test), dtype=float) for h in horizons}
    full_fill = {h: np.zeros(len(X_train), dtype=float) for h in horizons}
    num_round = int(params.pop("num_boost_round"))

    for seed in seeds:
        p = dict(params)
        p["seed"] = seed
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        for tr_idx, va_idx in cv.split(X_train, STRAT_LABELS):
            dtr = xgb.DMatrix(X_train.iloc[tr_idx], label=labels[tr_idx])
            dva = xgb.DMatrix(X_train.iloc[va_idx], label=labels[va_idx])
            booster = xgb.train(p, dtr, num_boost_round=num_round, evals=[(dtr, "train")], verbose_eval=False)

            score_tr = booster.predict(dtr, output_margin=True)
            score_va = booster.predict(dva, output_margin=True)

            for h in horizons:
                cal = fit_calibrator(score_tr, time_values[tr_idx], event_values[tr_idx], h)
                oof[h][va_idx] += cal.predict(score_va)
            cnt[va_idx] += 1.0

        dfull = xgb.DMatrix(X_train, label=labels)
        booster_full = xgb.train(p, dfull, num_boost_round=num_round, evals=[(dfull, "train")], verbose_eval=False)
        score_full = booster_full.predict(dfull, output_margin=True)
        score_test = booster_full.predict(xgb.DMatrix(X_test), output_margin=True)

        for h in horizons:
            cal_full = fit_calibrator(score_full, time_values, event_values, h)
            full_fill[h] += cal_full.predict(score_full)
            test_pred[h] += cal_full.predict(score_test)

    nz = cnt > 0
    out_oof, out_test = {}, {}
    for h in horizons:
        out_oof[h] = np.zeros(len(X_train), dtype=float)
        out_oof[h][nz] = oof[h][nz] / cnt[nz]
        out_oof[h][~nz] = full_fill[h][~nz] / len(seeds)
        out_test[h] = test_pred[h] / len(seeds)
        out_oof[h] = np.clip(out_oof[h], 0.0, 1.0)
        out_test[h] = np.clip(out_test[h], 0.0, 1.0)
    return out_oof, out_test

def repeated_xgb_aft(X_train, X_test, params, seeds, horizons=(12, 24, 48), pred_mode="margin", n_splits=5):
    lower = time_values.copy()
    upper = np.where(event_values == 1, time_values, np.inf)

    distribution = params["aft_loss_distribution"]
    scale = float(params["aft_loss_distribution_scale"])
    num_round = int(params.pop("num_boost_round"))

    oof = {h: np.zeros(len(X_train), dtype=float) for h in horizons}
    cnt = np.zeros(len(X_train), dtype=float)
    test_pred = {h: np.zeros(len(X_test), dtype=float) for h in horizons}
    full_fill = {h: np.zeros(len(X_train), dtype=float) for h in horizons}

    for seed in seeds:
        p = dict(params)
        p["seed"] = seed
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        for tr_idx, va_idx in cv.split(X_train, STRAT_LABELS):
            dtr = xgb.DMatrix(X_train.iloc[tr_idx])
            dva = xgb.DMatrix(X_train.iloc[va_idx])

            dtr.set_float_info("label_lower_bound", lower[tr_idx])
            dtr.set_float_info("label_upper_bound", upper[tr_idx])
            dva.set_float_info("label_lower_bound", lower[va_idx])
            dva.set_float_info("label_upper_bound", upper[va_idx])

            booster = xgb.train(p, dtr, num_boost_round=num_round, evals=[(dtr, "train")], verbose_eval=False)
            margin_tr = booster.predict(dtr, output_margin=True)
            margin_va = booster.predict(dva, output_margin=True)

            for h in horizons:
                if pred_mode == "cdf":
                    score_tr = aft_cdf_from_margin(margin_tr, h, distribution, scale)
                    score_va = aft_cdf_from_margin(margin_va, h, distribution, scale)
                else:
                    score_tr = -margin_tr
                    score_va = -margin_va
                cal = fit_calibrator(score_tr, time_values[tr_idx], event_values[tr_idx], h)
                oof[h][va_idx] += cal.predict(score_va)

            cnt[va_idx] += 1.0

        dfull = xgb.DMatrix(X_train)
        dfull.set_float_info("label_lower_bound", lower)
        dfull.set_float_info("label_upper_bound", upper)
        booster_full = xgb.train(p, dfull, num_boost_round=num_round, evals=[(dfull, "train")], verbose_eval=False)
        margin_full = booster_full.predict(dfull, output_margin=True)
        margin_test = booster_full.predict(xgb.DMatrix(X_test), output_margin=True)

        for h in horizons:
            if pred_mode == "cdf":
                score_full = aft_cdf_from_margin(margin_full, h, distribution, scale)
                score_test = aft_cdf_from_margin(margin_test, h, distribution, scale)
            else:
                score_full = -margin_full
                score_test = -margin_test
            cal_full = fit_calibrator(score_full, time_values, event_values, h)
            full_fill[h] += cal_full.predict(score_full)
            test_pred[h] += cal_full.predict(score_test)

    nz = cnt > 0
    out_oof, out_test = {}, {}
    for h in horizons:
        out_oof[h] = np.zeros(len(X_train), dtype=float)
        out_oof[h][nz] = oof[h][nz] / cnt[nz]
        out_oof[h][~nz] = full_fill[h][~nz] / len(seeds)
        out_test[h] = test_pred[h] / len(seeds)
        out_oof[h] = np.clip(out_oof[h], 0.0, 1.0)
        out_test[h] = np.clip(out_test[h], 0.0, 1.0)
    return out_oof, out_test

FAST_SEEDS = (123, 456, 789)
FULL_SEEDS = (123, 456, 789, 777, 666, 1511)
TREE_SEEDS = FAST_SEEDS if FAST else FULL_SEEDS
SURV_SEEDS = TREE_SEEDS[:4]
CAT_SEEDS = TREE_SEEDS[:3]

def make_logit_builder(C):
    def _builder(seed):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=C, max_iter=4000, solver="lbfgs"))
        ])
    return _builder

def make_lgb_builder(cfg):
    def _builder(seed):
        return lgb.LGBMClassifier(
            objective="binary",
            boosting_type="gbdt",
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            max_depth=cfg["max_depth"],
            num_leaves=cfg["num_leaves"],
            subsample=cfg["subsample"],
            colsample_bytree=cfg["colsample_bytree"],
            min_child_samples=cfg["min_child_samples"],
            reg_alpha=cfg["reg_alpha"],
            reg_lambda=cfg["reg_lambda"],
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    return _builder

def make_xgb_builder(cfg):
    def _builder(seed):
        return xgb.XGBClassifier(
            objective="binary:logistic",
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            max_depth=cfg["max_depth"],
            min_child_weight=cfg["min_child_weight"],
            subsample=cfg["subsample"],
            colsample_bytree=cfg["colsample_bytree"],
            reg_alpha=cfg["reg_alpha"],
            reg_lambda=cfg["reg_lambda"],
            gamma=cfg["gamma"],
            tree_method="hist",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1,
            verbosity=0,
        )
    return _builder

def make_cat_builder(cfg):
    def _builder(seed):
        return CatBoostClassifier(
            loss_function="Logloss",
            iterations=cfg["iterations"],
            learning_rate=cfg["learning_rate"],
            depth=cfg["depth"],
            l2_leaf_reg=cfg["l2_leaf_reg"],
            bootstrap_type=cfg["bootstrap_type"],
            subsample=cfg["subsample"],
            random_strength=cfg["random_strength"],
            verbose=False,
            random_seed=seed,
        )
    return _builder

global_lgb_cfgs = {
    12: {"n_estimators": 220, "learning_rate": 0.040, "max_depth": 2, "num_leaves": 4, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5},
    24: {"n_estimators": 280, "learning_rate": 0.035, "max_depth": 2, "num_leaves": 4, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 6, "reg_alpha": 0.8, "reg_lambda": 2.5},
    48: {"n_estimators": 320, "learning_rate": 0.030, "max_depth": 3, "num_leaves": 6, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0},
}

global_xgb_cfgs = {
    12: {"n_estimators": 180, "learning_rate": 0.040, "max_depth": 2, "min_child_weight": 2.0, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.4, "reg_lambda": 2.0, "gamma": 0.0},
    24: {"n_estimators": 220, "learning_rate": 0.035, "max_depth": 2, "min_child_weight": 3.0, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.6, "reg_lambda": 2.5, "gamma": 0.0},
    48: {"n_estimators": 260, "learning_rate": 0.028, "max_depth": 3, "min_child_weight": 4.0, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.8, "reg_lambda": 3.0, "gamma": 0.0},
}

global_cat_cfgs = {
    12: {"iterations": 260, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 10.0, "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    24: {"iterations": 300, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 12.0, "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    48: {"iterations": 340, "learning_rate": 0.028, "depth": 4, "l2_leaf_reg": 14.0, "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
}

zone_lgb_cfgs = {
    12: {"n_estimators": 200, "learning_rate": 0.045, "max_depth": 2, "num_leaves": 4, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 4, "reg_alpha": 0.4, "reg_lambda": 1.2},
    24: {"n_estimators": 220, "learning_rate": 0.040, "max_depth": 2, "num_leaves": 4, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5},
    48: {"n_estimators": 220, "learning_rate": 0.035, "max_depth": 2, "num_leaves": 4, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 5, "reg_alpha": 0.6, "reg_lambda": 1.8},
}

zone_xgb_cfgs = {
    12: {"n_estimators": 160, "learning_rate": 0.045, "max_depth": 2, "min_child_weight": 2.0, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.4, "reg_lambda": 1.5, "gamma": 0.0},
    24: {"n_estimators": 180, "learning_rate": 0.040, "max_depth": 2, "min_child_weight": 2.0, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.4, "reg_lambda": 1.5, "gamma": 0.0},
    48: {"n_estimators": 200, "learning_rate": 0.035, "max_depth": 2, "min_child_weight": 3.0, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.5, "reg_lambda": 2.0, "gamma": 0.0},
}

cox_params = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "tree_method": "hist",
    "learning_rate": 0.030,
    "max_depth": 2,
    "min_child_weight": 2.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "lambda": 2.0,
    "alpha": 0.2,
    "num_boost_round": 260 if not FAST else 180,
}

aft_normal_params = {
    "objective": "survival:aft",
    "eval_metric": "aft-nloglik",
    "aft_loss_distribution": "normal",
    "aft_loss_distribution_scale": 1.00,
    "tree_method": "hist",
    "learning_rate": 0.030,
    "max_depth": 2,
    "min_child_weight": 2.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "lambda": 2.0,
    "alpha": 0.2,
    "num_boost_round": 280 if not FAST else 180,
}

aft_logistic_params = {
    "objective": "survival:aft",
    "eval_metric": "aft-nloglik",
    "aft_loss_distribution": "logistic",
    "aft_loss_distribution_scale": 1.20,
    "tree_method": "hist",
    "learning_rate": 0.030,
    "max_depth": 2,
    "min_child_weight": 2.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "lambda": 2.0,
    "alpha": 0.2,
    "num_boost_round": 280 if not FAST else 180,
}

linear_cs = {12: 0.60, 24: 0.80, 48: 1.00}

ZONE_CFG = {
    12: {"center": 4200.0, "width": 1100.0, "near_pad": 2200.0, "far_pad": 1200.0},
    24: {"center": 5200.0, "width": 1500.0, "near_pad": 2600.0, "far_pad": 1400.0},
    48: {"center": 6500.0, "width": 1800.0, "near_pad": 3200.0, "far_pad": 1800.0},
}

y_map = {}
mask_map = {}
for h in [12, 24, 48]:
    y_map[h], mask_map[h] = make_binary_target(time_values, event_values, h)

near_train_masks = {}
far_train_masks = {}
for h in [12, 24, 48]:
    cfg = ZONE_CFG[h]
    near_train_masks[h] = mask_map[h] & ((dist_train_m < cfg["center"] + cfg["near_pad"]) | (y_map[h] == 1))
    far_train_masks[h] = mask_map[h] & ((dist_train_m >= cfg["center"] - cfg["far_pad"]) | (y_map[h] == 1))

experts = {}

def register_expert(name, oof_dict, test_dict):
    experts[name] = {"oof": oof_dict, "test": test_dict}

# anchors
anchor_inv_oof, anchor_inv_test = {}, {}
anchor_mix_oof, anchor_mix_test = {}, {}
for h in [12, 24, 48]:
    inv_signal_train = 1.0 / (train_proc["dist_km"].values + 0.25)
    inv_signal_test = 1.0 / (test_proc["dist_km"].values + 0.25)
    eta_signal_train = train_proc[f"eta_vs_{h}"].values if f"eta_vs_{h}" in train_proc.columns else 1.0 / (train_proc["eta_effective"].values / h + 1.0)
    eta_signal_test = test_proc[f"eta_vs_{h}"].values if f"eta_vs_{h}" in test_proc.columns else 1.0 / (test_proc["eta_effective"].values / h + 1.0)

    anchor_inv_oof[h], anchor_inv_test[h] = repeated_isotonic(inv_signal_train, inv_signal_test, y_map[h], mask_map[h], TREE_SEEDS, n_splits=5)
    anchor_eta_oof, anchor_eta_test = repeated_isotonic(eta_signal_train, eta_signal_test, y_map[h], mask_map[h], TREE_SEEDS, n_splits=5)

    anchor_mix_oof[h] = np.clip(0.85 * anchor_inv_oof[h] + 0.15 * anchor_eta_oof, 0.0, 1.0)
    anchor_mix_test[h] = np.clip(0.85 * anchor_inv_test[h] + 0.15 * anchor_eta_test, 0.0, 1.0)

register_expert("anchor_inv", anchor_inv_oof, anchor_inv_test)
register_expert("anchor_mix", anchor_mix_oof, anchor_mix_test)

# linear
linear_oof, linear_test = {}, {}
for h in [12, 24, 48]:
    linear_oof[h], linear_test[h] = repeated_binary_model(
        X_linear_train, X_linear_test, y_map[h], mask_map[h],
        make_logit_builder(linear_cs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    linear_oof[h] = stabilize(linear_oof[h], anchor_mix_oof[h], 0.96)
    linear_test[h] = stabilize(linear_test[h], anchor_mix_test[h], 0.96)
register_expert("linear", linear_oof, linear_test)

# global tree classifiers
global_lgb_oof, global_lgb_test = {}, {}
global_xgb_oof, global_xgb_test = {}, {}
global_cat_oof, global_cat_test = {}, {}
for h in [12, 24, 48]:
    global_lgb_oof[h], global_lgb_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_lgb_builder(global_lgb_cfgs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_xgb_oof[h], global_xgb_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_xgb_builder(global_xgb_cfgs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_cat_oof[h], global_cat_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_cat_builder(global_cat_cfgs[h]),
        CAT_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )

    global_lgb_oof[h] = stabilize(global_lgb_oof[h], anchor_mix_oof[h], 0.95)
    global_lgb_test[h] = stabilize(global_lgb_test[h], anchor_mix_test[h], 0.95)

    global_xgb_oof[h] = stabilize(global_xgb_oof[h], anchor_mix_oof[h], 0.95)
    global_xgb_test[h] = stabilize(global_xgb_test[h], anchor_mix_test[h], 0.95)

    global_cat_oof[h] = stabilize(global_cat_oof[h], anchor_mix_oof[h], 0.93)
    global_cat_test[h] = stabilize(global_cat_test[h], anchor_mix_test[h], 0.93)

register_expert("lgb_global", global_lgb_oof, global_lgb_test)
register_expert("xgb_global", global_xgb_oof, global_xgb_test)
register_expert("cat_global", global_cat_oof, global_cat_test)

# zone specialists
near_lgb_oof, near_lgb_test = {}, {}
far_lgb_oof, far_lgb_test = {}, {}
near_xgb_oof, near_xgb_test = {}, {}
far_xgb_oof, far_xgb_test = {}, {}

for h in [12, 24, 48]:
    near_lgb_oof[h], near_lgb_test[h] = repeated_binary_model(
        X_near_train, X_near_test, y_map[h], mask_map[h],
        make_lgb_builder(zone_lgb_cfgs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48]), train_select_mask=near_train_masks[h]
    )
    far_lgb_oof[h], far_lgb_test[h] = repeated_binary_model(
        X_far_train, X_far_test, y_map[h], mask_map[h],
        make_lgb_builder(zone_lgb_cfgs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48]), train_select_mask=far_train_masks[h]
    )
    near_xgb_oof[h], near_xgb_test[h] = repeated_binary_model(
        X_near_train, X_near_test, y_map[h], mask_map[h],
        make_xgb_builder(zone_xgb_cfgs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48]), train_select_mask=near_train_masks[h]
    )
    far_xgb_oof[h], far_xgb_test[h] = repeated_binary_model(
        X_far_train, X_far_test, y_map[h], mask_map[h],
        make_xgb_builder(zone_xgb_cfgs[h]),
        TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48]), train_select_mask=far_train_masks[h]
    )

    near_lgb_oof[h] = stabilize(near_lgb_oof[h], anchor_mix_oof[h], 0.95)
    near_lgb_test[h] = stabilize(near_lgb_test[h], anchor_mix_test[h], 0.95)
    far_lgb_oof[h] = stabilize(far_lgb_oof[h], anchor_mix_oof[h], 0.95)
    far_lgb_test[h] = stabilize(far_lgb_test[h], anchor_mix_test[h], 0.95)

    near_xgb_oof[h] = stabilize(near_xgb_oof[h], anchor_mix_oof[h], 0.95)
    near_xgb_test[h] = stabilize(near_xgb_test[h], anchor_mix_test[h], 0.95)
    far_xgb_oof[h] = stabilize(far_xgb_oof[h], anchor_mix_oof[h], 0.95)
    far_xgb_test[h] = stabilize(far_xgb_test[h], anchor_mix_test[h], 0.95)

register_expert("lgb_near", near_lgb_oof, near_lgb_test)
register_expert("lgb_far", far_lgb_oof, far_lgb_test)
register_expert("xgb_near", near_xgb_oof, near_xgb_test)
register_expert("xgb_far", far_xgb_oof, far_xgb_test)

# survival experts
try:
    cox_oof, cox_test = repeated_xgb_cox(X_surv_train, X_surv_test, dict(cox_params), SURV_SEEDS, horizons=(12, 24, 48), n_splits=5)
    for h in [12, 24, 48]:
        cox_oof[h] = stabilize(cox_oof[h], anchor_mix_oof[h], 0.96)
        cox_test[h] = stabilize(cox_test[h], anchor_mix_test[h], 0.96)
    register_expert("xgb_cox", cox_oof, cox_test)
except Exception as e:
    print("[WARN] xgb_cox failed:", repr(e))

try:
    aft_margin_oof, aft_margin_test = repeated_xgb_aft(X_surv_train, X_surv_test, dict(aft_normal_params), SURV_SEEDS, horizons=(12, 24, 48), pred_mode="margin", n_splits=5)
    for h in [12, 24, 48]:
        aft_margin_oof[h] = stabilize(aft_margin_oof[h], anchor_mix_oof[h], 0.96)
        aft_margin_test[h] = stabilize(aft_margin_test[h], anchor_mix_test[h], 0.96)
    register_expert("aft_norm_margin", aft_margin_oof, aft_margin_test)
except Exception as e:
    print("[WARN] aft_norm_margin failed:", repr(e))

try:
    aft_cdf_oof, aft_cdf_test = repeated_xgb_aft(X_surv_train, X_surv_test, dict(aft_logistic_params), SURV_SEEDS, horizons=(12, 24, 48), pred_mode="cdf", n_splits=5)
    for h in [12, 24, 48]:
        aft_cdf_oof[h] = stabilize(aft_cdf_oof[h], anchor_mix_oof[h], 0.96)
        aft_cdf_test[h] = stabilize(aft_cdf_test[h], anchor_mix_test[h], 0.96)
    register_expert("aft_logistic_cdf", aft_cdf_oof, aft_cdf_test)
except Exception as e:
    print("[WARN] aft_logistic_cdf failed:", repr(e))

def make_comparable_pairs(time, event):
    ii, jj = [], []
    for i in range(len(time)):
        if event[i] != 1:
            continue
        comp = np.where(time[i] < time)[0]
        comp = comp[comp != i]
        ii.extend([i] * len(comp))
        jj.extend(comp.tolist())
    return np.asarray(ii, dtype=int), np.asarray(jj, dtype=int)

PAIR_I, PAIR_J = make_comparable_pairs(time_values, event_values)

def soft_cindex(risk, temperature=0.05):
    if len(PAIR_I) == 0:
        return 0.5
    diff = (np.asarray(risk)[PAIR_I] - np.asarray(risk)[PAIR_J]) / max(temperature, 1e-6)
    return float(np.mean(expit(diff)))

def zone_eval_mask(h, zone):
    center = ZONE_CFG[h]["center"]
    if zone == "near":
        return mask_map[h] & (dist_train_m < center)
    return mask_map[h] & (dist_train_m >= center)

def select_group(h, zone, expert_names, top_k=7, temp=0.012):
    rows = []
    zmask = zone_eval_mask(h, zone)
    for name in expert_names:
        if name not in experts or h not in experts[name]["oof"]:
            continue
        p = experts[name]["oof"][h]
        if zmask.sum() == 0:
            b = compute_brier(time_values, event_values, p, h)
        else:
            b = float(np.mean((p[zmask] - y_map[h][zmask]) ** 2))
        rows.append((b, name))
    rows = sorted(rows, key=lambda x: x[0])
    if not rows:
        rows = [(0.0, "anchor_inv")]
    keep = rows[:min(top_k, len(rows))]
    names = [name for _, name in keep]
    P_oof = np.column_stack([experts[name]["oof"][h] for name in names])
    P_test = np.column_stack([experts[name]["test"][h] for name in names])
    bvals = np.array([b for b, _ in keep], dtype=float)
    if len(bvals) == 1:
        prior = np.ones(1, dtype=float)
    else:
        prior = np.exp(-(bvals - bvals.min()) / max(temp, 1e-4))
        prior = prior / prior.sum()
    return names, P_oof, P_test, prior

ALL_EXPERT_NAMES = list(experts.keys())
print("Experts:", ALL_EXPERT_NAMES)

def group_constraints(group_sizes):
    constraints = []
    start = 0
    spans = {}
    for key, size in group_sizes:
        spans[key] = (start, start + size)
        constraints.append({"type": "eq", "fun": lambda x, a=start, b=start + size: np.sum(x[a:b]) - 1.0})
        start += size
    return spans, constraints

def normalize_group_vector(x, spans, priors):
    x = np.asarray(x, dtype=float).copy()
    for key, (a, b) in spans.items():
        s = x[a:b].sum()
        if s <= 0:
            x[a:b] = priors[key]
        else:
            x[a:b] = np.clip(x[a:b], 0.0, None)
            x[a:b] /= x[a:b].sum()
    return x

def predict_group(P, w):
    if P.shape[1] == 1:
        return P[:, 0]
    return P @ w

def optimize_joint_24_48(groups, gate24, gate48, reg=0.035, shrink=0.75, temp=0.05):
    group_sizes = [(key, groups[key]["P_oof"].shape[1]) for key in ["24_near", "24_far", "48_near", "48_far"]]
    spans, constraints = group_constraints(group_sizes)
    priors = {key: groups[key]["prior"] for key in groups}
    x0 = np.concatenate([priors[key] for key in ["24_near", "24_far", "48_near", "48_far"]])
    bounds = [(0.0, 1.0)] * len(x0)

    def unpack(x, key):
        a, b = spans[key]
        w = np.asarray(x[a:b], dtype=float)
        s = w.sum()
        if s <= 0:
            return priors[key]
        w = np.clip(w, 0.0, None)
        return w / w.sum()

    def objective(x):
        x = normalize_group_vector(x, spans, priors)
        w24n = unpack(x, "24_near")
        w24f = unpack(x, "24_far")
        w48n = unpack(x, "48_near")
        w48f = unpack(x, "48_far")

        p24 = gate24 * predict_group(groups["24_near"]["P_oof"], w24n) + (1.0 - gate24) * predict_group(groups["24_far"]["P_oof"], w24f)
        p48 = gate48 * predict_group(groups["48_near"]["P_oof"], w48n) + (1.0 - gate48) * predict_group(groups["48_far"]["P_oof"], w48f)
        p48 = np.maximum(p48, p24)

        risk = 0.3 * p24 + 0.4 * p48 + 0.3
        c_soft = soft_cindex(risk, temperature=temp)
        b24 = float(np.mean((p24[mask_map[24]] - y_map[24][mask_map[24]]) ** 2))
        b48 = float(np.mean((p48[mask_map[48]] - y_map[48][mask_map[48]]) ** 2))

        reg_term = 0.0
        for key in priors:
            reg_term += float(np.sum((unpack(x, key) - priors[key]) ** 2))
        return -(0.3 * c_soft + 0.7 * (1.0 - (0.3 * b24 + 0.4 * b48))) + reg * reg_term

    try:
        res = minimize(
            objective, x0, method="SLSQP", bounds=bounds, constraints=constraints,
            options={"maxiter": 700, "ftol": 1e-10, "disp": False}
        )
        x_opt = normalize_group_vector(res.x if res.success else x0, spans, priors)
    except Exception:
        x_opt = normalize_group_vector(x0, spans, priors)

    x_final = shrink * x_opt + (1.0 - shrink) * x0
    x_final = normalize_group_vector(x_final, spans, priors)

    final_weights = {key: x_final[spans[key][0]:spans[key][1]] for key in priors}
    opt_weights = {key: x_opt[spans[key][0]:spans[key][1]] for key in priors}
    prior_weights = {key: x0[spans[key][0]:spans[key][1]] for key in priors}
    return final_weights, opt_weights, prior_weights

def predict_joint_24_48(groups, weights, gate24, gate48, split_kind):
    p24 = gate24 * predict_group(groups["24_near"][f"P_{split_kind}"], weights["24_near"]) + (1.0 - gate24) * predict_group(groups["24_far"][f"P_{split_kind}"], weights["24_far"])
    p48 = gate48 * predict_group(groups["48_near"][f"P_{split_kind}"], weights["48_near"]) + (1.0 - gate48) * predict_group(groups["48_far"][f"P_{split_kind}"], weights["48_far"])
    p48 = np.maximum(p48, p24)
    return np.clip(p24, 0.0, 1.0), np.clip(p48, 0.0, 1.0)

def optimize_12(groups, gate12, p24_ref, reg=0.040, shrink=0.75):
    group_sizes = [(key, groups[key]["P_oof"].shape[1]) for key in ["12_near", "12_far"]]
    spans, constraints = group_constraints(group_sizes)
    priors = {key: groups[key]["prior"] for key in groups}
    x0 = np.concatenate([priors[key] for key in ["12_near", "12_far"]])
    bounds = [(0.0, 1.0)] * len(x0)

    def unpack(x, key):
        a, b = spans[key]
        w = np.asarray(x[a:b], dtype=float)
        s = w.sum()
        if s <= 0:
            return priors[key]
        w = np.clip(w, 0.0, None)
        return w / w.sum()

    def objective(x):
        x = normalize_group_vector(x, spans, priors)
        w12n = unpack(x, "12_near")
        w12f = unpack(x, "12_far")
        p12 = gate12 * predict_group(groups["12_near"]["P_oof"], w12n) + (1.0 - gate12) * predict_group(groups["12_far"]["P_oof"], w12f)
        p12 = np.minimum(np.clip(p12, 0.0, 1.0), np.asarray(p24_ref, dtype=float))
        b12 = float(np.mean((p12[mask_map[12]] - y_map[12][mask_map[12]]) ** 2))
        reg_term = 0.0
        for key in priors:
            reg_term += float(np.sum((unpack(x, key) - priors[key]) ** 2))
        return b12 + reg * reg_term

    try:
        res = minimize(
            objective, x0, method="SLSQP", bounds=bounds, constraints=constraints,
            options={"maxiter": 500, "ftol": 1e-10, "disp": False}
        )
        x_opt = normalize_group_vector(res.x if res.success else x0, spans, priors)
    except Exception:
        x_opt = normalize_group_vector(x0, spans, priors)

    x_final = shrink * x_opt + (1.0 - shrink) * x0
    x_final = normalize_group_vector(x_final, spans, priors)

    final_weights = {key: x_final[spans[key][0]:spans[key][1]] for key in priors}
    opt_weights = {key: x_opt[spans[key][0]:spans[key][1]] for key in priors}
    prior_weights = {key: x0[spans[key][0]:spans[key][1]] for key in priors}
    return final_weights, opt_weights, prior_weights

def predict_12(groups, weights, gate12, split_kind, p24_ref):
    p12 = gate12 * predict_group(groups["12_near"][f"P_{split_kind}"], weights["12_near"]) + (1.0 - gate12) * predict_group(groups["12_far"][f"P_{split_kind}"], weights["12_far"])
    p12 = np.minimum(np.clip(p12, 0.0, 1.0), np.asarray(p24_ref, dtype=float))
    return p12

# build 24/48 groups
groups_24_48 = {}
weights_log = {}

for h in [24, 48]:
    center = ZONE_CFG[h]["center"]
    zone_candidates = [
        "anchor_inv", "anchor_mix", "linear",
        "lgb_global", "xgb_global", "cat_global",
        "lgb_near", "lgb_far", "xgb_near", "xgb_far",
        "xgb_cox", "aft_norm_margin", "aft_logistic_cdf",
    ]
    names_near, Pn_oof, Pn_test, prior_near = select_group(h, "near", zone_candidates, top_k=7, temp=0.012)
    names_far, Pf_oof, Pf_test, prior_far = select_group(h, "far", zone_candidates, top_k=7, temp=0.012)

    groups_24_48[f"{h}_near"] = {"names": names_near, "P_oof": Pn_oof, "P_test": Pn_test, "prior": prior_near}
    groups_24_48[f"{h}_far"] = {"names": names_far, "P_oof": Pf_oof, "P_test": Pf_test, "prior": prior_far}

gate24_train = soft_gate(dist_train_m, ZONE_CFG[24]["center"], ZONE_CFG[24]["width"])
gate24_test = soft_gate(dist_test_m, ZONE_CFG[24]["center"], ZONE_CFG[24]["width"])
gate48_train = soft_gate(dist_train_m, ZONE_CFG[48]["center"], ZONE_CFG[48]["width"])
gate48_test = soft_gate(dist_test_m, ZONE_CFG[48]["center"], ZONE_CFG[48]["width"])

joint_w_final, joint_w_opt, joint_w_prior = optimize_joint_24_48(groups_24_48, gate24_train, gate48_train, reg=0.035, shrink=0.75, temp=0.05)

for tag, wdict in [("final", joint_w_final), ("opt", joint_w_opt), ("prior", joint_w_prior)]:
    for key, value in wdict.items():
        weights_log[f"{key}_{tag}"] = {
            "names": groups_24_48[key]["names"],
            "weights": [float(x) for x in value]
        }

p24_prior_oof, p48_prior_oof = predict_joint_24_48(groups_24_48, joint_w_prior, gate24_train, gate48_train, "oof")
p24_prior_test, p48_prior_test = predict_joint_24_48(groups_24_48, joint_w_prior, gate24_test, gate48_test, "test")

p24_opt_oof, p48_opt_oof = predict_joint_24_48(groups_24_48, joint_w_final, gate24_train, gate48_train, "oof")
p24_opt_test, p48_opt_test = predict_joint_24_48(groups_24_48, joint_w_final, gate24_test, gate48_test, "test")

p24_safe_oof = np.clip(0.92 * p24_opt_oof + 0.08 * anchor_mix_oof[24], 0.0, 1.0)
p48_safe_oof = np.clip(np.maximum(0.94 * p48_opt_oof + 0.06 * anchor_mix_oof[48], p24_safe_oof), 0.0, 1.0)
p24_safe_test = np.clip(0.92 * p24_opt_test + 0.08 * anchor_mix_test[24], 0.0, 1.0)
p48_safe_test = np.clip(np.maximum(0.94 * p48_opt_test + 0.06 * anchor_mix_test[48], p24_safe_test), 0.0, 1.0)

candidate_24_48 = {
    "prior": (p24_prior_oof, p48_prior_oof, p24_prior_test, p48_prior_test),
    "opt": (p24_opt_oof, p48_opt_oof, p24_opt_test, p48_opt_test),
    "safe": (p24_safe_oof, p48_safe_oof, p24_safe_test, p48_safe_test),
}

best_name = None
best_score = -1.0
for name, (o24, o48, t24, t48) in candidate_24_48.items():
    hybrid, c_idx, wb, b24, b48, b72 = compute_hybrid_score(time_values, event_values, o24, o48, np.ones(len(train_df), dtype=float))
    weights_log[f"candidate_{name}"] = {
        "hybrid": float(hybrid), "cindex": float(c_idx),
        "wbrier": float(wb), "b24": float(b24), "b48": float(b48), "b72": float(b72)
    }
    print(f"{name:>6s} | Hybrid={hybrid:.6f} | C={c_idx:.6f} | B24={b24:.6f} | B48={b48:.6f}")
    if hybrid > best_score:
        best_score = hybrid
        best_name = name

p24_oof, p48_oof, p24_test, p48_test = candidate_24_48[best_name]
print("Chosen 24/48 blend:", best_name)

# build 12 groups after 24 is fixed
groups_12 = {}
zone_candidates_12 = [
    "anchor_inv", "anchor_mix", "linear",
    "lgb_global", "xgb_global", "cat_global",
    "lgb_near", "lgb_far", "xgb_near", "xgb_far",
    "xgb_cox", "aft_norm_margin", "aft_logistic_cdf",
]
names_12_near, P12n_oof, P12n_test, prior_12_near = select_group(12, "near", zone_candidates_12, top_k=7, temp=0.012)
names_12_far, P12f_oof, P12f_test, prior_12_far = select_group(12, "far", zone_candidates_12, top_k=7, temp=0.012)
groups_12["12_near"] = {"names": names_12_near, "P_oof": P12n_oof, "P_test": P12n_test, "prior": prior_12_near}
groups_12["12_far"] = {"names": names_12_far, "P_oof": P12f_oof, "P_test": P12f_test, "prior": prior_12_far}

gate12_train = soft_gate(dist_train_m, ZONE_CFG[12]["center"], ZONE_CFG[12]["width"])
gate12_test = soft_gate(dist_test_m, ZONE_CFG[12]["center"], ZONE_CFG[12]["width"])

w12_final, w12_opt, w12_prior = optimize_12(groups_12, gate12_train, p24_oof, reg=0.040, shrink=0.75)

for tag, wdict in [("final", w12_final), ("opt", w12_opt), ("prior", w12_prior)]:
    for key, value in wdict.items():
        weights_log[f"{key}_{tag}"] = {
            "names": groups_12[key]["names"],
            "weights": [float(x) for x in value]
        }

p12_prior_oof = predict_12(groups_12, w12_prior, gate12_train, "oof", p24_oof)
p12_prior_test = predict_12(groups_12, w12_prior, gate12_test, "test", p24_test)
p12_opt_oof = predict_12(groups_12, w12_final, gate12_train, "oof", p24_oof)
p12_opt_test = predict_12(groups_12, w12_final, gate12_test, "test", p24_test)
p12_safe_oof = np.minimum(np.clip(0.90 * p12_opt_oof + 0.10 * anchor_mix_oof[12], 0.0, 1.0), p24_oof)
p12_safe_test = np.minimum(np.clip(0.90 * p12_opt_test + 0.10 * anchor_mix_test[12], 0.0, 1.0), p24_test)

b12_prior = compute_brier(time_values, event_values, p12_prior_oof, 12)
b12_opt = compute_brier(time_values, event_values, p12_opt_oof, 12)
b12_safe = compute_brier(time_values, event_values, p12_safe_oof, 12)
print(f"12h prior Brier = {b12_prior:.6f}")
print(f"12h opt   Brier = {b12_opt:.6f}")
print(f"12h safe  Brier = {b12_safe:.6f}")

if b12_safe <= min(b12_opt, b12_prior):
    p12_oof, p12_test = p12_safe_oof, p12_safe_test
    chosen_12 = "safe"
elif b12_opt <= b12_prior:
    p12_oof, p12_test = p12_opt_oof, p12_opt_test
    chosen_12 = "opt"
else:
    p12_oof, p12_test = p12_prior_oof, p12_prior_test
    chosen_12 = "prior"

weights_log["chosen_12"] = chosen_12
weights_log["chosen_24_48"] = best_name

# final assembly
oof_final = np.column_stack([
    p12_oof,
    p24_oof,
    p48_oof,
    np.ones(len(train_df), dtype=float),
])
test_final = np.column_stack([
    p12_test,
    p24_test,
    p48_test,
    np.ones(len(test_df), dtype=float),
])

oof_final = enforce_monotonicity(oof_final)
test_final = enforce_monotonicity(test_final)

# optional legacy test-only hook
if USE_LEGACY_BLEND and LEGACY_FILES:
    legacy_mats = []
    search_roots = [WORK_DIR, DATA_DIR, "."]
    for item in LEGACY_FILES:
        cand_paths = [item] + [os.path.join(root, item) for root in search_roots]
        resolved = None
        for fp in cand_paths:
            if os.path.exists(fp):
                resolved = fp
                break
        if resolved is None:
            continue
        old = pd.read_csv(resolved)
        old = sample_sub[["event_id"]].merge(old, on="event_id", how="left", validate="one_to_one")
        legacy_mats.append(old[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values)
    if legacy_mats:
        legacy_mean = np.mean(np.stack(legacy_mats, axis=0), axis=0)
        test_final = enforce_monotonicity((1.0 - LEGACY_BLEND_WEIGHT) * test_final + LEGACY_BLEND_WEIGHT * legacy_mean)
        weights_log["legacy_blend"] = {
            "weight": float(LEGACY_BLEND_WEIGHT),
            "files": list(LEGACY_FILES),
            "n_loaded": int(len(legacy_mats)),
        }

if DO_OOF:
    hybrid, c_idx, wb, b24, b48, b72 = compute_hybrid_score(time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3])
    b12 = compute_brier(time_values, event_values, oof_final[:, 0], 12)
    print(f"OOF Hybrid = {hybrid:.6f}")
    print(f"OOF C-Index = {c_idx:.6f}")
    print(f"OOF WBrier = {wb:.6f}")
    print(f"OOF Brier@12 = {b12:.6f}")
    print(f"OOF Brier@24 = {b24:.6f}")
    print(f"OOF Brier@48 = {b48:.6f}")
    print(f"OOF Brier@72 = {b72:.6f}")

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": test_final[:, 0],
    "prob_24h": test_final[:, 1],
    "prob_48h": test_final[:, 2],
    "prob_72h": test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

def validate_submission(sub, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].nunique() == len(sub)
    vals = sub[required[1:]].values
    assert np.isfinite(vals).all()
    assert ((vals >= 0.0) & (vals <= 1.0)).all()
    assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
    assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
    assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

validate_submission(submission, sample_sub)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

if DO_OOF:
    oof_dump = pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_final[:, 0],
        "prob_24h": oof_final[:, 1],
        "prob_48h": oof_final[:, 2],
        "prob_72h": oof_final[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    })
    oof_dump.to_csv(os.path.join(WORK_DIR, "oof_predictions.csv"), index=False)

with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Saved:", OUTPUT_PATH)
print(submission.head())


DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 37) TEST = (95, 35)
Experts: ['anchor_inv', 'anchor_mix', 'linear', 'lgb_global', 'xgb_global', 'cat_global', 'lgb_near', 'lgb_far', 'xgb_near', 'xgb_far', 'xgb_cox', 'aft_norm_margin', 'aft_logistic_cdf']
 prior | Hybrid=0.971758 | C=0.937645 | B24=0.025603 | B48=0.014852
   opt | Hybrid=0.971968 | C=0.938225 | B24=0.025537 | B48=0.014773
  safe | Hybrid=0.971731 | C=0.937811 | B24=0.025695 | B48=0.015058
Chosen 24/48 blend: opt
12h prior Brier = 0.050999
12h opt   Brier = 0.050704
12h safe  Brier = 0.051040
OOF Hybrid = 0.971968
OOF C-Index = 0.938225
OOF WBrier = 0.013571
OOF Brier@12 = 0.050704
OOF Brier@24 = 0.025537
OOF Brier@48 = 0.014773
OOF Brier@72 = 0.000000
Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.008062  0.010041  0.010041       1.0
1  13353600  0.681063  0.953732  0.979693       1.0
2  13942327  0.008387  0.010649  0.010649      